# ELCVIA revision — detection results on Kaggle (Tables 2, 7, 8, 9)

Scores the 290 official UCF-Crime test videos and the 298 description videos with the pretrained C3D + Deep MIL detector.

**Before running**
1. The detector weights (`c3d.pickle`, `epoch_80000.pt`) are downloaded automatically from their public sources; no upload needed.
2. All links are already filled in (Part 1 is not used: 271 of the 290 test videos are scored). (open the UCF-Crime Dropbox folder → ⋯ next to each file → Copy link, and change the ending `dl=0` to `dl=1`).
3. Input: `elcvia-code`. Settings: **GPU T4 x2**, **Internet on**.
4. **Save Version → Save & Run All (Commit)**. If it stops early, attach this notebook's output as an extra input and run again; it continues.

At the end, download `det_results.zip` from the Output tab and upload it to Claude.

In [1]:
# 1) code, detector repo and weights
import os, glob, time, shutil, sys, json, inspect
START = time.time()
WORK = '/kaggle/working'; os.chdir(WORK)
def find_input(name, must_contain=None):
    hits = glob.glob(f'/kaggle/input/**/{name}', recursive=True)
    if must_contain: hits = [h for h in hits if must_contain in open(h, errors='ignore').read()]
    return hits[0] if hits else None
pipe = find_input('paper_results_pipeline.py', 'ft_train_step'); assert pipe, 'patched pipeline not found in inputs'
shutil.copy(pipe, WORK)
for f in ['revision_experiments.py', 'ftds_pairs.json']:
    p = find_input(f); assert p, f + ' not found'; shutil.copy(p, WORK)
REPO = '/kaggle/working/AnomalyDetectionCVPR2018-Pytorch'
if not os.path.isdir(REPO):
    os.system(f'git clone -q https://github.com/ekosman/AnomalyDetectionCVPR2018-Pytorch {REPO}')
# weights: taken from inputs if you uploaded them, otherwise downloaded from the public sources
C3D = find_input('c3d.pickle')
if not C3D:
    C3D = f'{REPO}/pretrained/c3d.pickle'; os.makedirs(os.path.dirname(C3D), exist_ok=True)
    if not os.path.exists(C3D) or os.path.getsize(C3D) < 100e6:
        os.system(f'wget -q -O {C3D} http://imagelab.ing.unimore.it/files/c3d_pytorch/c3d.pickle')
AD = find_input('epoch_80000.pt') or f'{REPO}/exps/c3d/models/epoch_80000.pt'
if not os.path.exists(AD) or os.path.getsize(AD) < 1e6:
    os.makedirs(os.path.dirname(AD), exist_ok=True)
    os.system(f'wget -q -O {AD} https://raw.githubusercontent.com/ekosman/AnomalyDetectionCVPR2018-Pytorch/main/exps/c3d/models/epoch_80000.pt')
assert os.path.getsize(C3D) > 100e6, 'c3d.pickle download failed - upload it as a dataset instead'
assert os.path.getsize(AD) > 1e6, 'epoch_80000.pt download failed - upload it as a dataset instead'
print('C3D size %.0f MB, AD size %.1f MB' % (os.path.getsize(C3D) / 1e6, os.path.getsize(AD) / 1e6))
print('C3D weights:', C3D, '\nAD weights:', AD)
sys.path.insert(0, REPO)
from utils.utils import build_transforms
print('---- C3D preprocessing used (for Section 3.1) ----'); print(inspect.getsource(build_transforms))
# continue from a previous run of this notebook
prev = glob.glob('/kaggle/input/**/results_det/frame_counts.json', recursive=True)
if prev and not os.path.isdir(f'{WORK}/results_det'):
    shutil.copytree(os.path.dirname(prev[0]), f'{WORK}/results_det'); print('continuing from', os.path.dirname(prev[0]))
!nvidia-smi --query-gpu=name,memory.total --format=csv

C3D size 320 MB, AD size 8.5 MB
C3D weights: /kaggle/working/AnomalyDetectionCVPR2018-Pytorch/pretrained/c3d.pickle 
AD weights: /kaggle/working/AnomalyDetectionCVPR2018-Pytorch/exps/c3d/models/epoch_80000.pt
---- C3D preprocessing used (for Section 3.1) ----
def build_transforms(mode: str = "c3d") -> transforms.Compose:
    """Build transforms to use for training an anomaly detection model.

    Args:
        mode (str, optional): Mode for which transforms should be constructed.
        Either c3d | i3d | mfnet | 3dResNet | r2plus1d | s3d | mvit. Defaults to "c3d".

    Raises:
        NotImplementedError: The provided mode is not implemented.

    Returns:
        transforms.Compose
    """
    if mode == "c3d":
        mean = [124 / 255, 117 / 255, 104 / 255]
        std = [1 / (0.0167 * 255)] * 3
        resize = 128, 171
        crop = 112
    elif mode == "i3d":
        mean = [0, 0, 0]
        std = [1, 1, 1]
    elif mode == "mfnet":
        mean = [0.485, 0.456, 0.406]
       

In [2]:
# 2) UCF-Crime videos (Parts 2-4 + normal test videos; Part 1 = Abuse/Arrest/Arson/Assault is not used)
#    -> 271 of the 290 official test videos are scored (19 Part-1 test videos excluded)
LINKS = {
 'part2': 'https://www.dropbox.com/scl/fo/2aczdnx37hxvcfdo4rq4q/ABYl45nk5Wd0ecKkP7VfnmI/Anomaly-Videos-Part-2.zip?rlkey=5bg7mxxbq46t7aujfch46dlvz&dl=1',
 'part3': 'https://www.dropbox.com/scl/fo/2aczdnx37hxvcfdo4rq4q/AFb9SoyBX4rJ-XrdqZDVA_I/Anomaly-Videos-Part-3.zip?rlkey=5bg7mxxbq46t7aujfch46dlvz&dl=1',
 'part4': 'https://www.dropbox.com/scl/fo/2aczdnx37hxvcfdo4rq4q/AAmA_Jr2it3xfS0o1J-WF1Q/Anomaly-Videos-Part-4.zip?rlkey=5bg7mxxbq46t7aujfch46dlvz&dl=1',
 'normal_test': 'https://www.dropbox.com/scl/fo/2aczdnx37hxvcfdo4rq4q/AHTTsM4LmY3lQdqO9mMBjFw/Testing_Normal_Videos.zip?rlkey=5bg7mxxbq46t7aujfch46dlvz&st=1szv6obk&dl=1',
}
import zipfile, subprocess, re

# --- official test list + temporal annotations, from the detector repository (290 videos)
ann = [l.split() for l in open(f'{REPO}/Test_Annotation.txt') if l.strip()]
TEST_LIST, GT = f'{WORK}/Anomaly_Test.txt', f'{WORK}/Temporal_Anomaly_Annotation_for_Testing_Videos.txt'
with open(TEST_LIST, 'w') as f:
    f.write('\n'.join(a[0] for a in ann) + '\n')
with open(GT, 'w') as f:      # official format: name  class  start1 end1 start2 end2
    f.write('\n'.join(f'{os.path.basename(a[0])}  {a[2]}  {a[3]}  {a[4]}  {a[5]}  {a[6]}' for a in ann) + '\n')
test_names = {os.path.splitext(os.path.basename(a[0]))[0] for a in ann}
ftds_names = set(json.load(open(f'{WORK}/ftds_pairs.json')).keys())
NEED = test_names | ftds_names
print('test videos listed:', len(test_names), '| description videos:', len(ftds_names), '| distinct videos needed:', len(NEED))

# --- download each zip, keep ONLY the needed videos, delete the zip at once
def writable_dirs():
    out = []
    for d in ['/kaggle/tmp', '/tmp', '/var/tmp', '/root', '/kaggle/working']:
        try:
            os.makedirs(d, exist_ok=True); t = f'{d}/.wtest'
            open(t, 'w').write('x'); os.remove(t); out.append((shutil.disk_usage(d).free, d))
        except OSError:
            print('not writable:', d)
    return sorted(out, reverse=True)
dirs = writable_dirs()
print('writable folders (free GB):', [(round(f / 1e9), d) for f, d in dirs])
assert dirs and dirs[0][1] != '/kaggle/working', ('the session disk is read-only - stop the session '
       '(power button, top right) and start it again, or use Run > Factory reset, then run all cells again')
VID = f'{dirs[0][1]}/videos'
os.makedirs(VID, exist_ok=True)
have = lambda: {os.path.splitext(os.path.basename(p))[0] for p in glob.glob(f'{VID}/**/*.mp4', recursive=True)}
for k, url in LINKS.items():
    if os.path.exists(f'{VID}/.{k}_done'): continue
    if 'PASTE' in url: print(f'!!! {k}: link not pasted - skipped'); continue
    head = subprocess.run(['curl', '-sIL', url], capture_output=True, text=True).stdout
    name = (re.findall(r'filename="?([^";\r\n]+)', head) or ['?'])[-1]
    size = int((re.findall(r'(?i)content-length:\s*(\d+)', head) or ['0'])[-1])
    free = shutil.disk_usage(os.path.dirname(VID)).free
    print(f'{k}: file "{name}", {size/1e9:.1f} GB, free disk {free/1e9:.0f} GB')
    if size and size * 1.1 > free:
        print(f'!!! {k}: not enough disk space'); continue
    t0, before = time.time(), have()
    z = f'{os.path.dirname(VID)}/part.zip'
    os.system(f'wget -q -O {z} "{url}"')
    print(f'   downloaded {os.path.getsize(z)/1e9:.1f} GB' if os.path.exists(z) else '   download failed')
    try:
        with zipfile.ZipFile(z) as zf:
            for m in zf.namelist():
                if m.endswith('.mp4') and os.path.splitext(os.path.basename(m))[0] in NEED:
                    zf.extract(m, VID)
    except zipfile.BadZipFile:
        print(f'!!! {k}: download is not a valid zip')
    if os.path.exists(z): os.remove(z)
    added = len(have() - before)
    print(f'   {k}: {added} needed videos kept in {(time.time()-t0)/60:.0f} min')
    if added: open(f'{VID}/.{k}_done', 'w').close()

missing = sorted(NEED - have())
miss_test = [v for v in missing if v in test_names]
print('needed videos on disk:', len(NEED) - len(missing), '/', len(NEED), '| test videos missing:', len(miss_test), miss_test[:8])
print('videos per part folder:', {d: len(glob.glob(f'{VID}/{d}/**/*.mp4', recursive=True)) for d in os.listdir(VID) if not d.startswith('.')})
assert len(NEED) - len(missing) > 0, 'NO videos were downloaded - send the output of this cell to Claude (nothing else will run)'

test videos listed: 290 | description videos: 298 | distinct videos needed: 516
writable folders (free GB): [(1149, '/tmp'), (1149, '/kaggle/tmp'), (1149, '/var/tmp'), (1149, '/root'), (20, '/kaggle/working')]
part2: file "unspecified", 6.7 GB, free disk 1149 GB
   downloaded 6.7 GB
   part2: 84 needed videos kept in 2 min
part3: file "unspecified", 5.6 GB, free disk 1146 GB
   downloaded 5.6 GB
   part3: 203 needed videos kept in 1 min
part4: file "unspecified", 6.6 GB, free disk 1143 GB
   downloaded 6.6 GB
   part4: 60 needed videos kept in 2 min
normal_test: file "unspecified", 4.7 GB, free disk 1140 GB
   downloaded 4.7 GB
   normal_test: 150 needed videos kept in 1 min
needed videos on disk: 497 / 516 | test videos missing: 19 ['Abuse028_x264', 'Abuse030_x264', 'Arrest001_x264', 'Arrest007_x264', 'Arrest024_x264', 'Arrest030_x264', 'Arrest039_x264', 'Arson007_x264']
videos per part folder: {'Anomaly-Videos-Part-3': 203, 'Anomaly-Videos-Part-4': 60, 'Testing_Normal_Videos_Anomaly'

In [3]:
# 3) configure the pipeline
# PyTorch >= 2.6 refuses to load full pickled models by default. The detector file comes from the
# official public repository (ekosman/AnomalyDetectionCVPR2018-Pytorch), so we load it the classic way.
import torch
if not getattr(torch.load, '_elcvia_patched', False):
    _orig_load = torch.load
    def _load(*a, **k):
        k.setdefault('weights_only', False)
        return _orig_load(*a, **k)
    _load._elcvia_patched = True
    torch.load = _load
sys.path.insert(0, WORK)
import paper_results_pipeline as P
P.CONFIG.update(repo=REPO + '/', c3d_weights=C3D, ad_weights=AD, ucf_root=VID + '/',
                test_list=TEST_LIST, gt=GT, pairs=f'{WORK}/ftds_pairs.json', ftds=f'{WORK}/ftds_pairs.json',
                out=f'{WORK}/results_det', frame_stride=1, transform='repo',
                deadline=START + 11.3 * 3600)
P.OUT = P.CONFIG['out']; os.makedirs(P.OUT, exist_ok=True)
_kc = P.known_counts
def _safe_counts():                      # frame counts come from the videos themselves; skip if the file format differs
    try: return _kc()
    except Exception as e: print('annotation frame counts skipped:', e); return {}
P.known_counts = _safe_counts
class A:
    def __init__(self, **k): self.__dict__.update(k)
missing = [v for v in P.split_names('test') if not P.find_video(v)]
print('test videos missing on disk:', len(missing), missing[:5])

test videos missing on disk: 19 ['Arson022_x264', 'Arrest039_x264', 'Arson018_x264', 'Arson011_x264', 'Assault011_x264']


In [4]:
# 4) score videos within a fixed time budget (saves after every chunk; nothing is lost if time runs out)
import torch, random
if not getattr(torch.load, '_elcvia_patched', False):          # PyTorch >= 2.6 fix (official public checkpoint)
    _orig_load = torch.load
    def _load(*a, **k):
        k.setdefault('weights_only', False); return _orig_load(*a, **k)
    _load._elcvia_patched = True; torch.load = _load

BUDGET_H = 3.3          # hours since the notebook started; stop scoring after this and bundle the results
_orig_split_names = P.split_names
def run_chunked(split, names, chunk=25):
    done = set(json.load(open(f'{P.OUT}/scores_{split}.json'))) if os.path.exists(f'{P.OUT}/scores_{split}.json') else set()
    names = [v for v in names if v not in done]
    print(f'{split}: {len(done)} already scored, {len(names)} to go')
    for i in range(0, len(names), chunk):
        if time.time() - START > BUDGET_H * 3600:
            print(f'time budget reached - stopping {split} early'); break
        sub = names[i:i + chunk]
        P.split_names = lambda s, sub=sub: sub
        try:
            P.stage_detect(A(split=split))
        finally:
            P.split_names = _orig_split_names
        print(f'   {split}: {min(i + chunk, len(names))}/{len(names)} done, {(time.time() - START) / 3600:.2f} h elapsed', flush=True)

# (a) the 60 description test videos -> Table 9 (small, done first)
desc_test = json.load(open(find_input('ftds_video_split.json')))['test']
run_chunked('ftds', [v for v in desc_test if P.find_video(v)])

# (b) the official test videos that are on disk -> Tables 2, 7, 8 (random order, so a partial run stays balanced)
test_vids = [v for v in P.split_names('test') if P.find_video(v)]
random.Random(0).shuffle(test_vids)
run_chunked('test', test_vids)

for s in ['ftds', 'test']:
    f = f'{P.OUT}/scores_{s}.json'
    print(s, 'scored:', len(json.load(open(f))) if os.path.exists(f) else 0)

ftds: 0 already scored, 60 to go
[1/25] Fighting006_x264: max=0.999 frames=944 3.1s
[2/25] Fighting011_x264: max=0.990 frames=8240 20.1s
[3/25] Fighting013_x264: max=0.994 frames=1434 3.0s
[4/25] Fighting018_x264: max=0.999 frames=1389 2.8s
[5/25] Fighting021_x264: max=0.983 frames=3567 7.4s
[6/25] Fighting025_x264: max=0.994 frames=4781 10.0s
[7/25] Fighting027_x264: max=0.999 frames=3851 8.2s
[8/25] Fighting028_x264: max=0.949 frames=2423 5.2s
[9/25] Fighting031_x264: max=0.055 frames=9001 19.6s
[10/25] Fighting047_x264: max=0.035 frames=4459 9.7s
[11/25] RoadAccidents002_x264: max=0.990 frames=347 0.8s
[12/25] RoadAccidents003_x264: max=0.995 frames=1483 3.3s
[13/25] RoadAccidents004_x264: max=0.278 frames=389 0.8s
[14/25] RoadAccidents006_x264: max=0.978 frames=1691 3.7s
[15/25] RoadAccidents013_x264: max=0.329 frames=1729 3.9s
[16/25] RoadAccidents014_x264: max=0.520 frames=11895 27.5s
[17/25] RoadAccidents016_x264: max=0.048 frames=2192 5.3s
[18/25] RoadAccidents023_x264: max=0.9

In [5]:
# 5) bundle for Claude (small)
os.chdir(WORK)
files = [f'results_det/{f}' for f in ['scores_test.json', 'scores_ftds.json', 'frame_counts.json', 'timing.json', 'frame_counts.csv']
         if os.path.exists(f'results_det/{f}')] + [os.path.basename(GT)]
os.system('rm -f det_results.zip && zip -q det_results.zip ' + ' '.join(files))
!ls -lh det_results.zip
print(f'run time: {(time.time()-START)/3600:.1f} h')

-rw-r--r-- 1 root root 114K Oct 11 08:07 det_results.zip
run time: 0.9 h
